# 3.5 Ontologías en la Web Semántica y grafos de conocimiento

**Asignatura:** Introducción a la Inteligencia Artificial  
**Unidad 3:** Representación del conocimiento y razonamiento

## Propósito

Construir un pequeño grafo de conocimiento académico usando **RDFLib**, representarlo mediante triples RDF y realizar consultas con **SPARQL**.

Al finalizar este notebook podrás:

a) Crear un grafo RDF

b) Definir recursos mediante un espacio de nombres

c) Agregar clases, propiedades e instancias

d) Representar relaciones mediante triples

e) Serializar el grafo en Turtle

f) Realizar consultas SPARQL

g) Observar una inferencia sencilla basada en RDFS


## 0. Preparación del entorno

Instalaremos:

- `rdflib` para crear y consultar grafos RDF
- `owlrl` para demostrar inferencia básica RDFS/OWL RL

En Google Colab, ejecuta la siguiente celda.


In [ ]:
!pip -q install rdflib owlrl

## 1. Importar bibliotecas y crear el grafo


In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, Literal
from rdflib.namespace import XSD

g = Graph()

print("Grafo creado.")
print("Número inicial de triples:", len(g))


## 2. Definir un espacio de nombres

Usaremos un espacio de nombres ficticio para el ejemplo:

```text
http://ejemplo.org/
```

Esto permitirá crear identificadores como:

```text
http://ejemplo.org/Ana
http://ejemplo.org/IntroduccionIA
http://ejemplo.org/cursa
```


In [ ]:
EX = Namespace("http://ejemplo.org/")

g.bind("ex", EX)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)

print(EX.Ana)
print(EX.IntroduccionIA)
print(EX.cursa)


## 3. Definir clases del dominio

Nuestro pequeño dominio académico tendrá las clases:

```text
Estudiante
EstudiantePosgrado
Profesor
Asignatura
Programa
Institucion
```


In [ ]:
clases = [
    EX.Estudiante,
    EX.EstudiantePosgrado,
    EX.Profesor,
    EX.Asignatura,
    EX.Programa,
    EX.Institucion
]

for clase in clases:
    g.add((clase, RDF.type, RDFS.Class))

# Jerarquía
g.add((EX.EstudiantePosgrado, RDFS.subClassOf, EX.Estudiante))

print("Clases agregadas.")
print("Número de triples:", len(g))


## 4. Definir propiedades

Utilizaremos las relaciones:

```text
cursa
imparte
formaParteDe
perteneceA
```

También añadiremos `domain` y `range` para mostrar cómo RDFS puede aportar semántica.


In [ ]:
propiedades = [
    EX.cursa,
    EX.imparte,
    EX.formaParteDe,
    EX.perteneceA
]

for prop in propiedades:
    g.add((prop, RDF.type, RDF.Property))

# Dominio y rango
g.add((EX.cursa, RDFS.domain, EX.Estudiante))
g.add((EX.cursa, RDFS.range, EX.Asignatura))

g.add((EX.imparte, RDFS.domain, EX.Profesor))
g.add((EX.imparte, RDFS.range, EX.Asignatura))

g.add((EX.formaParteDe, RDFS.domain, EX.Asignatura))
g.add((EX.formaParteDe, RDFS.range, EX.Programa))

g.add((EX.perteneceA, RDFS.domain, EX.Programa))
g.add((EX.perteneceA, RDFS.range, EX.Institucion))

print("Propiedades definidas.")


## 5. Crear instancias

Crearemos los siguientes recursos:

```text
Ana
Luis
Profesor1
IntroduccionIA
BasesDatos
MSC
TecNM
```


In [ ]:
g.add((EX.Ana, RDF.type, EX.EstudiantePosgrado))
g.add((EX.Luis, RDF.type, EX.Estudiante))

g.add((EX.Profesor1, RDF.type, EX.Profesor))

g.add((EX.IntroduccionIA, RDF.type, EX.Asignatura))
g.add((EX.BasesDatos, RDF.type, EX.Asignatura))

g.add((EX.MSC, RDF.type, EX.Programa))
g.add((EX.TecNM, RDF.type, EX.Institucion))

print("Instancias agregadas.")


## 6. Agregar hechos como triples RDF

Ahora representaremos conocimiento concreto:

```text
Ana cursa IntroduccionIA
Luis cursa BasesDatos
Profesor1 imparte IntroduccionIA
IntroduccionIA formaParteDe MSC
BasesDatos formaParteDe MSC
MSC perteneceA TecNM
```


In [ ]:
g.add((EX.Ana, EX.cursa, EX.IntroduccionIA))
g.add((EX.Luis, EX.cursa, EX.BasesDatos))
g.add((EX.Profesor1, EX.imparte, EX.IntroduccionIA))
g.add((EX.IntroduccionIA, EX.formaParteDe, EX.MSC))
g.add((EX.BasesDatos, EX.formaParteDe, EX.MSC))
g.add((EX.MSC, EX.perteneceA, EX.TecNM))

print("Número total de triples:", len(g))


## 7. Agregar propiedades literales

Un objeto RDF también puede ser un valor literal.


In [ ]:
g.add((EX.Ana, EX.nombre, Literal("Ana")))
g.add((EX.IntroduccionIA, EX.creditos, Literal(6, datatype=XSD.integer)))

print("Propiedades literales agregadas.")


## 8. Inspeccionar algunos triples

Cada triple tiene:

```text
Sujeto - Predicado - Objeto
```


In [ ]:
for i, (s, p, o) in enumerate(g):
    print(s, " | ", p, " | ", o)
    if i >= 12:
        break


## 9. Serializar el grafo en Turtle

Turtle permite escribir los triples de manera legible.


In [ ]:
turtle = g.serialize(format="turtle")
print(turtle)


## 10. Consulta SPARQL 1

Pregunta:

> ¿Qué estudiantes cursan `IntroduccionIA`?


In [ ]:
consulta_1 = '''
PREFIX ex: <http://ejemplo.org/>

SELECT ?estudiante
WHERE {
    ?estudiante ex:cursa ex:IntroduccionIA .
}
'''

for fila in g.query(consulta_1):
    print(fila.estudiante)


## 11. Consulta SPARQL 2

Pregunta:

> ¿Qué profesor imparte `IntroduccionIA`?


In [ ]:
consulta_2 = '''
PREFIX ex: <http://ejemplo.org/>

SELECT ?profesor
WHERE {
    ?profesor ex:imparte ex:IntroduccionIA .
}
'''

for fila in g.query(consulta_2):
    print(fila.profesor)


## 12. Consulta SPARQL 3

Pregunta:

> ¿A qué programa pertenece cada asignatura?


In [ ]:
consulta_3 = '''
PREFIX ex: <http://ejemplo.org/>

SELECT ?asignatura ?programa
WHERE {
    ?asignatura ex:formaParteDe ?programa .
}
'''

for fila in g.query(consulta_3):
    print("Asignatura:", fila.asignatura)
    print("Programa:", fila.programa)
    print()


## 13. Consulta SPARQL 4: relaciones encadenadas

Pregunta:

> ¿A qué institución pertenece el programa del que forma parte la asignatura cursada por Ana?

Patrón conceptual:

```text
Ana
 ↓ cursa
Asignatura
 ↓ formaParteDe
Programa
 ↓ perteneceA
Institucion
```


In [ ]:
consulta_4 = '''
PREFIX ex: <http://ejemplo.org/>

SELECT ?institucion
WHERE {
    ex:Ana ex:cursa ?asignatura .
    ?asignatura ex:formaParteDe ?programa .
    ?programa ex:perteneceA ?institucion .
}
'''

for fila in g.query(consulta_4):
    print("Institución:", fila.institucion)


## 14. Mundo abierto

Supongamos que preguntamos si Luis cursa `IntroduccionIA`.

El grafo **no contiene** ese triple.

En un enfoque de mundo abierto, la ausencia del triple no implica automáticamente que la afirmación sea falsa.

Solo significa que no tenemos esa información en el grafo.


In [ ]:
triple_busqueda = (EX.Luis, EX.cursa, EX.IntroduccionIA)

if triple_busqueda in g:
    print("El grafo contiene la afirmación.")
else:
    print("El grafo no contiene la afirmación.")
    print("Bajo mundo abierto, esto no equivale automáticamente a negarla.")


## 15. Inferencia RDFS

Hasta ahora hemos almacenado explícitamente:

```text
Ana rdf:type EstudiantePosgrado
```

y también:

```text
EstudiantePosgrado rdfs:subClassOf Estudiante
```

Un razonador RDFS puede inferir:

```text
Ana rdf:type Estudiante
```

Usaremos `owlrl` para aplicar el cierre semántico.


In [ ]:
from owlrl import DeductiveClosure, RDFS_Semantics

g_inferido = Graph()

for triple in g:
    g_inferido.add(triple)

DeductiveClosure(RDFS_Semantics).expand(g_inferido)

print("Triples antes de inferencia:", len(g))
print("Triples después de inferencia:", len(g_inferido))

triple_inferido = (EX.Ana, RDF.type, EX.Estudiante)

print(
    "¿Se puede inferir que Ana es Estudiante?",
    triple_inferido in g_inferido
)


## 16. Comparar conocimiento explícito e inferido


In [ ]:
print("En el grafo original:")
print((EX.Ana, RDF.type, EX.Estudiante) in g)

print("\nDespués de aplicar inferencia RDFS:")
print((EX.Ana, RDF.type, EX.Estudiante) in g_inferido)


## 17. Experimento 1

Agrega una nueva estudiante:

```text
Maria
```

y representa:

```text
Maria cursa IntroduccionIA
Maria pertenece a MSC
```

Después modifica la consulta SPARQL para recuperar:

```text
Ana
Maria
```

como estudiantes que cursan `IntroduccionIA`.


In [ ]:
# Escribe aquí tu solución

# g.add(...)
# g.add(...)

# consulta = '''
# PREFIX ex: <http://ejemplo.org/>
# SELECT ?estudiante
# WHERE {
#     ...
# }
# '''


## 18. Experimento 2

Agrega una nueva clase:

```text
AsignaturaPosgrado
```

y establece que:

```text
AsignaturaPosgrado
rdfs:subClassOf
Asignatura
```

Después define:

```text
IntroduccionIA rdf:type AsignaturaPosgrado
```

Aplica nuevamente la inferencia y verifica si también puede inferirse:

```text
IntroduccionIA rdf:type Asignatura
```


In [ ]:
# Escribe aquí tu solución


## 19. Experimento 3: diseñar una consulta

Formula una consulta SPARQL para responder:

> ¿Qué estudiantes cursan asignaturas que forman parte de la MSC?

Pista:

```text
?estudiante ── cursa ──► ?asignatura
?asignatura ── formaParteDe ──► MSC
```


In [ ]:
# Escribe aquí tu consulta SPARQL


## 20. Preguntas de reflexión

a) ¿Qué diferencia existe entre una entidad y una clase?

b) ¿Qué representa un triple RDF?

c) ¿Por qué un conjunto de triples puede verse como un grafo?

d) ¿Qué diferencia existe entre RDF y RDFS?

e) ¿Qué función cumple SPARQL?

f) ¿Qué diferencia existe entre consultar un grafo e inferir nuevo conocimiento?

g) ¿Por qué la ausencia de un triple no implica necesariamente que sea falso?

h) ¿Qué parte del ejemplo corresponde a la ontología y qué parte al grafo de conocimiento?


## 21. Conclusión

En este notebook construimos un pequeño grafo de conocimiento académico y recorrimos la secuencia:

```text
Entidades
   ↓
IRI
   ↓
Triples RDF
   ↓
Grafo
   ↓
RDFS
   ↓
Inferencia
   ↓
SPARQL
   ↓
Consulta
```

La idea central es:

> **La ontología define la estructura del dominio; el grafo de conocimiento representa entidades y hechos concretos; SPARQL permite consultar ese conocimiento.**

Este ejercicio constituye una base para construir grafos más complejos en dominios reales.
